# LightGBM model

LightGBM is another boosted-tree library, like CatBoost, but it builds trees differently:

- **Leaf-wise growth.** CatBoost grows symmetric trees level by level. LightGBM keeps splitting whichever leaf reduces the error most, so its trees are lopsided and can find patterns CatBoost misses. That also makes it easier to overfit, so we keep leaves small and large.
- **Different category handling.** Rather than encoding each category as a default rate, it searches for the best way to split the categories into two groups.

Because it works differently, its mistakes may differ from CatBoost's, which is what makes it useful in a blend. Run this in your `dsde` environment.

## 0. Reload shared code automatically

Your features live in `features.py`, which every model notebook shares. Python normally loads a file like that only once per kernel, so later edits would be ignored.

`autoreload` makes the notebook pick up your latest `features.py` each time a cell runs. Keep this as the first code cell.

In [1]:
%load_ext autoreload
%autoreload 2

## Project folders

This notebook lives in `notebooks/`, but the data, shared code and outputs are in other folders. This cell finds the project folder and imports their locations from `scripts/paths.py`:

| Name | Folder | Holds |
|---|---|---|
| `DATA` | `data/` | Competition files |
| `PREDS` | `preds_for_blend/` | Saved predictions for blending |
| `SUBMISSIONS` | `submissions/` | Files to upload |
| `LOGS` | `logs/` | CatBoost training logs |

It also adds `scripts/` to Python's search path, so `from features import ...` works.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))

from paths import DATA, LOGS, PREDS, SUBMISSIONS

## 1. Imports

`lightgbm` is the model library. The rest are the same tools as in your CatBoost notebook.

In [2]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from features import CANDIDATES, add_features

## 2. Settings

`SEED = 42` and 5 folds match your other notebooks. That's essential: the blend only works if every model is tested on **exactly the same folds**, so their out-of-fold predictions are comparable.

In [3]:
TARGET = "12-Month Default Flag"
SEED = 42
N_SPLITS = 5
PRED_DIR = PREDS
PRED_DIR.mkdir(exist_ok=True)

## 3. Load the data

The same as in your CatBoost notebook: `X` has the features, `y` has the 0/1 flag, and `X_test` has the same columns in the same order.

Never sort or filter rows here. The saved predictions must stay in the original row order, or the blend would line up the wrong people.

**Features:** `add_features()` from `features.py` adds your engineered columns. It runs on train and test in exactly the same way, so they always have the same columns. Here, `X_test` takes `X`'s column list, which keeps the order identical.


In [4]:
train = pd.read_csv(DATA / "train_dataset.csv")
test = pd.read_csv(DATA / "test_without_gt.csv")

X = add_features(train.drop(columns=["ID", TARGET]))
y = train[TARGET]
X_test = add_features(test)[X.columns]

cats = [
    "Application Channel", "Occupation Category", "Industry Category",
    "Employment Type Category", "Insurance Type", "Company Size Category",
    "Business Entity Type", "Residence Type", "Residence Ownership Status",
]

nums = [c for c in X.columns if c not in cats]
print(X.shape, X_test.shape)

(68546, 26) (137091, 26)


## 4. Mark the category columns

LightGBM recognizes pandas' `category` type as a category column.

For the test set, we reuse **the training set's list of categories**. LightGBM stores categories by position in that list, so train and test must use the same list, or code 140 could mean different things in each. Any test code that train never saw becomes missing, which LightGBM can handle.

In [5]:
X_lgb = X.copy()
X_test_lgb = X_test.copy()
for c in cats:
    X_lgb[c] = X_lgb[c].astype("category")
    X_test_lgb[c] = pd.Categorical(X_test_lgb[c], categories=X_lgb[c].cat.categories)

/var/folders/8_/lk3rtfx97cl0flh0n1fn10100000gn/T/ipykernel_63964/4060679651.py:5: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_test_lgb[c] = pd.Categorical(X_test_lgb[c], categories=X_lgb[c].cat.categories)


## 5. Define the model

These settings come from trial 51 of `tuning_loop.py`, which scored 0.6520 mean CV AUC against 0.6497 for the previous settings, and won on all 5 folds.

| Setting | What it does |
|---|---|
| `n_estimators=5000` | The **maximum** number of trees. Early stopping usually stops far sooner, at about 160 trees here. |
| `learning_rate=0.03` | How much each tree contributes. In tuning, 0.01 scored the same but needed about 3 times as many trees. |
| `num_leaves=51` | Up to 51 leaves per tree. |
| `min_child_samples=833` | Each leaf must contain at least 833 people. This, more than the leaf count, keeps the trees from fitting noise. |
| `subsample=0.85`, `subsample_freq=1` | Each tree trains on a random 85% of the rows, which adds variety and reduces overfitting. |
| `colsample_bytree=0.78` | Each tree sees a random 78% of the features. |
| `reg_lambda=0.004` | L2 penalty on leaf values. It's almost switched off. |
| `reg_alpha=0.76` | L1 penalty on leaf values. |
| `cat_smooth=20.6` | Pulls category statistics toward the overall average, so rare categories aren't trusted too much. |
| `min_data_per_group=340` | A category value needs about 340 rows before it can get its own branch. |
| `verbose=-1` | Hides training messages. |

In [6]:
def make_lgbm(n_estimators=5000):
    # Tuned settings: trial 51 from tuning_loop.py (CV AUC 0.6520).
    # Previous: learning_rate=0.01, num_leaves=8, min_child_samples=100,
    # subsample=0.8, colsample_bytree=0.7, reg_lambda=5 (CV AUC 0.6497).
    return lgb.LGBMClassifier(
        n_estimators=n_estimators,
        learning_rate=0.03,
        num_leaves=51,
        min_child_samples=833,
        subsample=0.8503,
        subsample_freq=1,
        colsample_bytree=0.7785,
        reg_lambda=0.004063,
        reg_alpha=0.7572,
        cat_smooth=20.60,
        min_data_per_group=340,
        random_state=SEED,
        verbose=-1,
    )

## 6. Cross-validate with early stopping

The same loop as in your CatBoost notebook, so the two are directly comparable.

- `eval_X`, `eval_y` and `eval_metric="auc"` let LightGBM check validation AUC after each tree.
- `lgb.early_stopping(200)` stops when AUC hasn't improved for 200 trees and keeps the best tree count.
- `model.best_iteration_` gives that count.

As with CatBoost, the validation fold helps pick the tree count, so these scores are slightly optimistic.

In [7]:
cv = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
oof = np.zeros(len(y))
scores, best_iters = [], []

for fold, (fit_idx, val_idx) in enumerate(cv.split(X_lgb, y)):
    X_fit, y_fit = X_lgb.iloc[fit_idx], y.iloc[fit_idx]
    X_val, y_val = X_lgb.iloc[val_idx], y.iloc[val_idx]

    model = make_lgbm()
    model.fit(
        X_fit, y_fit,
        eval_X=X_val,
        eval_y=y_val,
        eval_metric="auc",
        callbacks=[lgb.early_stopping(200, verbose=False)],
    )

    oof[val_idx] = model.predict_proba(X_val)[:, 1]
    scores.append(roc_auc_score(y_val, oof[val_idx]))
    best_iters.append(model.best_iteration_)
    print(f"Fold {fold}: AUC {scores[-1]:.4f}, best trees {best_iters[-1]}")

print(f"\nLightGBM CV AUC: {np.mean(scores):.4f} +/- {np.std(scores):.4f}")

Fold 0: AUC 0.6488, best trees 175
Fold 1: AUC 0.6693, best trees 127
Fold 2: AUC 0.6456, best trees 178
Fold 3: AUC 0.6571, best trees 235
Fold 4: AUC 0.6391, best trees 107

LightGBM CV AUC: 0.6520 +/- 0.0104


## 7. Which features matter

`importance_type="gain"` measures how much each feature's splits reduced the error in total. We convert it to percentages so it's comparable with CatBoost's importance.

If the two models rank features very differently, that's a sign they're learning different things, which is good for blending.

In [8]:
gain = pd.Series(
    model.booster_.feature_importance(importance_type="gain"), index=X_lgb.columns
)
(gain / gain.sum() * 100).sort_values(ascending=False).round(2).head(12)

Age (in months)                  15.49
debt_to_income                    7.54
Company Size Category             7.39
Declared Annual Income            7.19
Occupation Category               7.01
Declared Number of Borrowings     7.01
Desired Credit Limit              6.02
Industry Category                 5.94
Residence Period (in months)      5.30
Residence Type                    3.98
Application Channel               3.90
Declared Borrowing Amount         3.45
dtype: float64

## 8. Train the final model on all the data

As before: the average best tree count from CV, times 1.1 because the full dataset is 25% larger than each training fold.

In [9]:
final_iters = int(np.mean(best_iters) * 1.1)
print("Trees for the final model:", final_iters)

final_model = make_lgbm(n_estimators=final_iters)
final_model.fit(X_lgb, y)
test_pred = final_model.predict_proba(X_test_lgb)[:, 1]

Trees for the final model: 180


## Write the submission

The `assert` lines stop the notebook if the file breaks the competition's format, so you don't waste one of your 10 daily submissions on a bad upload.

In [10]:
sub = pd.read_csv(DATA / "submission_template_for_test.csv")
assert (sub["ID"].values == test["ID"].values).all(), "Row order doesn't match the template"

sub[TARGET] = test_pred

assert list(sub.columns) == ["ID", TARGET]
assert len(sub) == 137_091
assert sub[TARGET].between(0, 1).all() and sub[TARGET].notna().all()

sub.to_csv(SUBMISSIONS / "submission_lgbm.csv", index=False)
print("Saved submissions/submission_lgbm.csv")

Saved submission_lgbm.csv


## Save predictions for the blend

This is what `06_blend.ipynb` reads. It needs both files:
- `lgbm_oof.npy`: one honest prediction per **training** row, used to score the blend
- `lgbm_test.npy`: one prediction per **test** row, used to build the blended submission

These are derived from competition data, so delete the `preds_for_blend` folder after the competition.

In [11]:
np.save(PRED_DIR / "lgbm_oof.npy", oof)
np.save(PRED_DIR / "lgbm_test.npy", test_pred)
print("Saved preds_for_blend/lgbm_oof.npy and preds_for_blend/lgbm_test.npy")

Saved preds_for_blend/lgbm_oof.npy and preds_for_blend/lgbm_test.npy


# Feature experiments

This section tests feature ideas. It doesn't change your submission or saved predictions, so you can run it as often as you like.

The idea: score the current features, then add **one** candidate at a time and score again on the **same folds**. A feature earns its place only if it helps consistently, not just on average.

We use LightGBM here because it's fast: each test takes about 12 seconds, while CatBoost takes about 1.5 minutes.

## A. Investigate the outliers first

Ratios divide one column by another, so a single absurd value can wreck them. An income of ¥94.8 billion would make that person's debt-to-income almost zero.

Compare the median, the 99th and 99.9th percentiles, and the maximum. A huge jump from the 99.9th percentile to the maximum suggests data-entry or unit errors rather than real people.

This prints summary statistics only, with no customer rows.

In [12]:
outlier_cols = ["Declared Annual Income", "Declared Borrowing Amount", "Rent Burden", "Desired Credit Limit"]
X[outlier_cols].quantile([0.5, 0.99, 0.999, 1.0]).T.rename(columns={1.0: "max"})

,0.5,0.99,0.999,max
Declared Annual Income,3000000.0,10000000.0,24805550.0,1.000000e+08
Declared Borrowing Amount,300000.0,4000000.0,9000000.0,1.236661e+10
Rent Burden,10000.0,130000.0,250000.0,1.000000e+06
Desired Credit Limit,200000.0,3000000.0,8000000.0,8.000000e+06


## B. A quick scoring function

`quick_cv()` is your cross-validation loop, shortened to return just the 5 fold scores. It uses `make_lgbm()` from above, with the same early stopping.

The `seed` setting chooses the folds. Keep the default for comparisons, because both the baseline and the new version must use the same folds.

In [13]:
def quick_cv(X_try, seed=SEED):
    folds = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed)
    s = []
    for fit_idx, val_idx in folds.split(X_try, y):
        m = make_lgbm()
        m.fit(
            X_try.iloc[fit_idx], y.iloc[fit_idx],
            eval_X=X_try.iloc[val_idx], eval_y=y.iloc[val_idx],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )
        s.append(roc_auc_score(y.iloc[val_idx], m.predict_proba(X_try.iloc[val_idx])[:, 1]))
    return np.array(s)

base = quick_cv(X_lgb)
print(f"Baseline: {base.mean():.4f}  folds {base.round(4)}")

Baseline: 0.6520  folds [0.6488 0.6693 0.6456 0.6571 0.6391]


## C. Test each candidate

The candidates are defined in `CANDIDATES` in `features.py`. For each one, this:
1. copies the current features,
2. adds that one new column,
3. scores it on the same folds, and
4. compares it with the baseline.

The new column is computed from `X`, the raw table, because the formulas use plain numbers rather than LightGBM's category columns. `X` and `X_lgb` have the same rows in the same order.

**How to read the table:**
- `change` is the difference in mean AUC from the baseline.
- `wins` is how many of the 5 folds improved.
- Keep a feature only if `change` is positive **and** `wins` is at least 4. Changes under about 0.002 are usually noise.

This takes about 1.5 minutes.

In [14]:
rows = []
for name, make_col in CANDIDATES.items():
    if name in X.columns:
        print(f"Skipping {name}: already in add_features()")
        continue
    X_try = X_lgb.copy()
    X_try[name] = make_col(X)
    new = quick_cv(X_try)
    rows.append({
        "feature": name,
        "cv_auc": new.mean(),
        "change": new.mean() - base.mean(),
        "wins": int((new > base).sum()),
    })
    print(f"{name}: {new.mean():.4f} ({new.mean() - base.mean():+.4f}), wins {rows[-1]['wins']} of 5")

pd.DataFrame(rows).sort_values("change", ascending=False).round(4)

Skipping debt_to_income: already in add_features()
total_ask_to_income: 0.6515 (-0.0005), wins 2 of 5
limit_to_debt: 0.6508 (-0.0012), wins 1 of 5
limit_to_income: 0.6506 (-0.0014), wins 1 of 5
rent_to_income: 0.6508 (-0.0011), wins 1 of 5
avg_loan: 0.6508 (-0.0012), wins 1 of 5
age_at_move_in: 0.6510 (-0.0009), wins 2 of 5
residence_share: 0.6509 (-0.0010), wins 1 of 5
unknown_count: 0.6506 (-0.0013), wins 1 of 5


,feature,cv_auc,change,wins
0,total_ask_to_income,0.6515,-0.0005,2
5,age_at_move_in,0.6510,-0.0009,2
6,residence_share,0.6509,-0.0010,1
3,rent_to_income,0.6508,-0.0011,1
4,avg_loan,0.6508,-0.0012,1
1,limit_to_debt,0.6508,-0.0012,1
7,unknown_count,0.6506,-0.0013,1
2,limit_to_income,0.6506,-0.0014,1


## D. Double-check a winner on different folds

A feature can look good on one set of folds by luck. Before keeping it, rescore the baseline and the new version on a **different** fold split, set by `seed=7`.

If it still helps, it's more likely to be real. Change `FEATURE` to the candidate you want to check.

In [15]:
FEATURE = "unknown_count"

X_try = X_lgb.copy()
X_try[FEATURE] = CANDIDATES[FEATURE](X)

base_7 = quick_cv(X_lgb, seed=7)
new_7 = quick_cv(X_try, seed=7)
print(f"{FEATURE} on seed 7: change {new_7.mean() - base_7.mean():+.4f}, wins {(new_7 > base_7).sum()} of 5")

unknown_count on seed 7: change -0.0004, wins 1 of 5


## E. Keep the winners

For each feature that passes both checks, add a line to `add_features()` in `features.py`:

```python
df["debt_to_income"] = CANDIDATES["debt_to_income"](df)
```

Then:
1. **Rerun this notebook from the top.** The baseline now includes the kept features, so the remaining candidates are tested on top of them. Features can overlap, so a second one may stop helping once the first is in.
2. **Rerun the other model notebooks**, then `06_blend`, so every model and the blend use the same features.
3. **Record each test** in your results log, including features you rejected.

**Adding your own idea:** add a new entry to `CANDIDATES` in `features.py`, then rerun section C.